# Lab 3: เพิ่มแท็บ "แนะนำการรดน้ำ" เข้าแอปเดิมของ Lab 2
### วิชา 11504233 ปัญญาประดิษฐ์สำหรับเกษตรอัจฉริยะ ·

> **โน้ตบุ๊กฉบับ VS Code** — เปิดด้วย VS Code แล้วรันทีละเซลล์จากบนลงล่าง

## Lab นี้ทำอะไร

เทรนโมเดล Machine Learning ที่ตอบว่า "วันนี้ควรรดน้ำไหม" แล้ว **เอาขึ้นเว็บจริง**
โดยเพิ่มเป็น **แท็บที่ 4** ในเว็บแอปที่ deploy ไว้แล้วตั้งแต่ Lab 2

| | ทำแบบนี้ |
|---|---|
| ต้องสมัครสมาชิกใหม่ | **ไม่ต้อง** ใช้บัญชี Streamlit Cloud เดิมจาก Lab 2 |
| ต้องติดตั้งอะไรเพิ่ม | **ไม่ต้อง** มี streamlit อยู่แล้วจาก Lab 2 |
| ใช้ Python 3.9 ได้ไหม | **ได้** ไม่มีเงื่อนไขเรื่องเวอร์ชัน |
| ผลลัพธ์ | **แท็บที่ 4 ในเว็บเดิม** — ระบบเดียวครบ |

เกษตรกรจึงได้เว็บเดียวที่ดูอากาศ ระดับน้ำ ราคา และคำแนะนำการรดน้ำได้ครบ
ไม่ต้องเปิดหลายลิงก์ — ตรงกับการใช้งานจริงมากกว่าการทำเว็บแยกทีละอัน

## แผนของโน้ตบุ๊กนี้
1. โหลดข้อมูลอากาศจริง 1 ปีจากไฟล์ CSV แล้วเทรน Random Forest
2. วัดผลด้วย train/test split และ cross-validation
3. เพิ่มโค้ด 3 ก้อนเข้า `deploy/app.py` ให้กลายเป็นแท็บที่ 4
4. รันเว็บดูในเครื่อง
5. push ขึ้น GitHub → Streamlit Cloud อัปเดตให้เอง

## ขั้นที่ 0: ติดตั้งไลบรารี

In [36]:
import sys, subprocess

pkgs = ["streamlit>=1.30,<2.0", "pandas>=2.2,<3.0",
        "scikit-learn>=1.6,<2.0", "requests"]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

v = sys.version_info
print(f"Python {v.major}.{v.minor}.{v.micro}")
import streamlit, sklearn, pandas
print("streamlit", streamlit.__version__,
      "| scikit-learn", sklearn.__version__,
      "| pandas", pandas.__version__)

Python 3.9.6
streamlit 1.50.0 | scikit-learn 1.6.1 | pandas 2.3.3


## ขั้นที่ 1: โหลดข้อมูลจากไฟล์ CSV

ข้อมูลอยู่ที่ `lab03-water-1y.csv` (โฟลเดอร์เดียวกับโน้ตบุ๊ก) — **อากาศจริงย้อนหลัง 1 ปี (365 วัน)**
ดึงจาก Open-Meteo Archive API ที่พิกัด ม.แม่โจ้ (18.90, 99.01)

| คอลัมน์ | ความหมาย | ที่มา |
|---|---|---|
| `moisture` | ความชื้นดิน 0-7 ซม. (%) | วัดจริง (ERA5) |
| `temp` | อุณหภูมิสูงสุด (°C) | วัดจริง |
| `rain` | ฝนของวันถัดไป (มม.) = "ฝนพยากรณ์" | วัดจริง |
| `rain3` | ฝนสะสม 3 วันจบวันนี้ (มม.) | คำนวณจากค่าจริง |
| `humid` | ความชื้นอากาศเฉลี่ย (%) | วัดจริง |
| `et0` | การคายระเหยอ้างอิง (มม./วัน) | คำนวณจริง (FAO) |
| `water` | **คำตอบ** 1 = ควรรดน้ำ, 0 = ไม่ต้องรด | **กฎเกษตร ไม่ใช่บันทึกจริง** |

> **ตรงไปตรงมาเรื่องข้อมูล:** ตัวแปรต้น (X) ทั้ง 6 ตัวเป็นค่าที่วัดจริง แต่ **ป้ายคำตอบ (y)
> สร้างจากกฎ** ว่า "ดินแห้ง + ฝนไม่มา + พืชคายน้ำสูง = ควรรด" เพราะไม่มีบันทึกการรดน้ำจริง
> ของสวน โมเดลจึงเก่งได้แค่เท่าที่กฎถูก — ถ้าจะใช้จริงต้องเก็บบันทึกการรดน้ำของสวนเอง

In [37]:
import pandas as pd

df = pd.read_csv("lab03-water-1y.csv")
คุณลักษณะ = ["moisture", "temp", "rain", "rain3", "humid", "et0"]

X = df[คุณลักษณะ]
y = df["water"]

print("ข้อมูล", df.shape[0], "แถว", len(คุณลักษณะ), "features")
print("ช่วงวันที่", df["date"].min(), "ถึง", df["date"].max())
print("คลาส:", y.value_counts().to_dict(), "(1 = ควรรดน้ำ)")
df.head()

ข้อมูล 365 แถว 6 features
ช่วงวันที่ 2025-08-08 ถึง 2026-08-07
คลาส: {0: 233, 1: 132} (1 = ควรรดน้ำ)


,date,moisture,temp,rain,rain3,humid,et0,water
0,2025-08-08,41.8,29.3,9.4,56.9,88,3.04,0
1,2025-08-09,39.4,31.0,17.1,38.6,86,4.21,0
2,2025-08-10,39.3,31.2,11.2,39.7,84,4.48,0
3,2025-08-11,39.1,31.3,3.2,37.7,84,4.28,0
4,2025-08-12,36.3,30.8,11.0,31.5,85,4.52,0


ลองดูว่าแต่ละเดือนต้องรดน้ำกี่วัน — จะเห็นฤดูกาลชัดเจน

In [38]:
รายเดือน = df.assign(เดือน=df["date"].str[:7]).groupby("เดือน")["water"].agg(["sum", "count"])
รายเดือน.columns = ["วันที่ต้องรด", "จำนวนวัน"]
print(รายเดือน.to_string())

         วันที่ต้องรด  จำนวนวัน
เดือน                          
2025-08             0        24
2025-09             0        30
2025-10             8        31
2025-11             5        30
2025-12            30        31
2026-01            11        31
2026-02             6        28
2026-03            29        31
2026-04            29        30
2026-05            11        31
2026-06             3        30
2026-07             0        31
2026-08             0         7


หน้าแล้ง (ธ.ค.–เม.ย.) ต้องรดเกือบทุกวัน ส่วนหน้าฝน (มิ.ย.–ก.ย.) แทบไม่ต้องรดเลย
— นี่คือรูปแบบที่โมเดลจะเรียนรู้ และเป็นเหตุผลที่ต้องใช้ข้อมูล **ครบทั้งปี**
ถ้าเก็บแค่ 90 วันช่วงหน้าฝน จะได้ตัวอย่าง "ควรรดน้ำ" แค่ 9% ซึ่งน้อยเกินไปจนโมเดลเรียนไม่ได้
(ปัญหา **class imbalance** ในบทที่ 4)

## ขั้นที่ 2: เทรนและวัดผล

ข้อมูล 365 แถวมากพอที่จะ **แบ่ง train/test** ได้จริง (ตอนมี 12 แถวทำไม่ได้)

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

#model = RandomForestClassifier(n_estimators=200, random_state=42)
model = KNeighborsClassifier(n_neighbors=3, metric="euclidean")
model.fit(X_tr, y_tr)

print("Test accuracy : %.3f" % model.score(X_te, y_te))
cv = cross_val_score(model, X, y, cv=5)
print("Cross-val (5)  : %.3f  เบี่ยงเบน %.3f" % (cv.mean(), cv.std()))
print()
print(classification_report(y_te, model.predict(X_te),
                            target_names=["ไม่ต้องรด", "ควรรดน้ำ"], digits=3))

Test accuracy : 0.957
Cross-val (5)  : 0.849  เบี่ยงเบน 0.078

              precision    recall  f1-score   support

   ไม่ต้องรด      0.951     0.983     0.967        59
    ควรรดน้ำ      0.968     0.909     0.938        33

    accuracy                          0.957        92
   macro avg      0.959     0.946     0.952        92
weighted avg      0.957     0.957     0.956        92



### ปัจจัยไหนสำคัญที่สุด

In [40]:
# น้ำหนัก = pd.Series(model.feature_importances_, index=คุณลักษณะ)
# print(น้ำหนัก.sort_values(ascending=False).round(3).to_string())

ความชื้นดินมีน้ำหนักมากที่สุด ตรงกับสามัญสำนึกของชาวสวน — เป็นสัญญาณว่าโมเดลเรียนรู้
สิ่งที่สมเหตุสมผล ไม่ได้จับสัญญาณมั่ว

In [41]:
def ทำนาย(ความชื้นดิน, อุณหภูมิ, ฝนพรุ่งนี้, ฝน3วัน, ความชื้นอากาศ, การคายระเหย):
    x = pd.DataFrame([[ความชื้นดิน, อุณหภูมิ, ฝนพรุ่งนี้,
                       ฝน3วัน, ความชื้นอากาศ, การคายระเหย]], columns=คุณลักษณะ)
    ผล = int(model.predict(x)[0])
    มั่นใจ = float(model.predict_proba(x)[0][ผล])
    คำตอบ = "ควรรดน้ำ" if ผล == 1 else "ไม่ต้องรดน้ำ (ดินชื้นพอ/ฝนจะตก)"
    return f"{คำตอบ}  (ความมั่นใจ {มั่นใจ:.0%})"


print("หน้าแล้ง ดินแห้ง ไม่มีฝน ->", ทำนาย(15, 35, 0, 0, 45, 5.0))
print("หน้าฝน ดินชื้น ฝนกำลังมา ->", ทำนาย(40, 28, 25, 60, 90, 2.5))

หน้าแล้ง ดินแห้ง ไม่มีฝน -> ควรรดน้ำ  (ความมั่นใจ 100%)
หน้าฝน ดินชื้น ฝนกำลังมา -> ไม่ต้องรดน้ำ (ดินชื้นพอ/ฝนจะตก)  (ความมั่นใจ 100%)


ผลที่ต้องได้

| สถานการณ์ | ผลที่ต้องได้ |
|---|---|
| ดิน 15%, ร้อน 35°C, ไม่มีฝน, อากาศแห้ง | **ควรรดน้ำ** |
| ดิน 40%, 28°C, ฝนพรุ่งนี้ 25 มม. | **ไม่ต้องรดน้ำ** |

## ขั้นที่ 3: รันเว็บดูในเครื่อง

เซลล์ถัดไปเปิดเซิร์ฟเวอร์ไว้เบื้องหลัง แล้วบอก URL ให้

In [44]:
import subprocess, sys, time, os

APP = os.path.join("deploy", "app.py")

เซิร์ฟเวอร์ = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP,
     "--server.headless=true", "--server.port=8501", "--server.address=localhost"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(8)                       # รอให้เซิร์ฟเวอร์ตั้งตัว
print("เปิดเว็บที่   http://localhost:8501")
print("ไปที่แท็บที่ 4 ชื่อ 'แนะนำการรดน้ำ'")
print()
print("วิธีทดสอบ: ติ๊กออกจาก 'ดึงค่าอากาศที่เหลือจาก API จริง' เพื่อกรอกเอง แล้วลอง")
print("   ดิน 15%  อุณหภูมิ 35  ฝนพรุ่งนี้ 0   -> ต้องได้ 'ควรรดน้ำ'")
print("   ดิน 40%  อุณหภูมิ 28  ฝนพรุ่งนี้ 25  -> ต้องได้ 'ไม่ต้องรดน้ำ'")
print()
print("(กด Ctrl+คลิกที่ลิงก์เพื่อเปิดในเบราว์เซอร์ — รันเซลล์ถัดไปเมื่อจะปิด)")

เปิดเว็บที่   http://localhost:8501
ไปที่แท็บที่ 4 ชื่อ 'แนะนำการรดน้ำ'

วิธีทดสอบ: ติ๊กออกจาก 'ดึงค่าอากาศที่เหลือจาก API จริง' เพื่อกรอกเอง แล้วลอง
   ดิน 15%  อุณหภูมิ 35  ฝนพรุ่งนี้ 0   -> ต้องได้ 'ควรรดน้ำ'
   ดิน 40%  อุณหภูมิ 28  ฝนพรุ่งนี้ 25  -> ต้องได้ 'ไม่ต้องรดน้ำ'

(กด Ctrl+คลิกที่ลิงก์เพื่อเปิดในเบราว์เซอร์ — รันเซลล์ถัดไปเมื่อจะปิด)


In [43]:
# รันเซลล์นี้เมื่อต้องการปิดเซิร์ฟเวอร์
เซิร์ฟเวอร์.terminate()
print("ปิดเซิร์ฟเวอร์แล้ว")

ปิดเซิร์ฟเวอร์แล้ว


> **รันจาก Terminal แทนก็ได้** (บางคนถนัดกว่า เห็น log ชัดกว่า)
> ```
> streamlit run deploy/app.py
> ```
> หยุดด้วย `Ctrl + C`

## ขั้นที่ 4: Deploy — ไม่ต้องสมัครอะไรใหม่เลย

แอปนี้ deploy อยู่บน **Streamlit Cloud** ตั้งแต่ Lab 1-4 แล้ว การอัปเดตจึงเหลือแค่ push โค้ด

1. เอา 2 ไฟล์ในโฟลเดอร์ **`deploy/`** (`app.py` และ `requirements.txt`) ขึ้น GitHub repo เดิม
   ```
   git add app.py requirements.txt
   git commit -m "เพิ่มแท็บแนะนำการรดน้ำ (Lab 5)"
   git push
   ```
2. Streamlit Cloud เห็นการ push แล้ว **rebuild ให้เองภายใน 1-2 นาที**
3. เปิดลิงก์เดิมที่เคยส่งอาจารย์ไว้ จะเห็นแท็บที่ 4 เพิ่มขึ้นมา

ขั้นตอนละเอียด (รวมกรณียังไม่เคย deploy) อยู่ใน `deploy/README-deploy.md`

> ครั้งแรกหลังเพิ่ม `scikit-learn` เข้า `requirements.txt` จะ build นานกว่าปกติหน่อย (~2-3 นาที)
> เพราะต้องติดตั้งไลบรารีใหม่

### ปัญหาที่พบบ่อย

| อาการ | วิธีแก้ |
|---|---|
| `ModuleNotFoundError: sklearn` บน Streamlit Cloud | ลืมเพิ่ม `scikit-learn` ใน `deploy/requirements.txt` แล้ว push |
| เว็บช้ามากเวลาขยับสไลเดอร์ | ลืมใส่ `@st.cache_resource` บนฟังก์ชันเทรนโมเดล |
| `NameError: แท็บรดน้ำ` | ยังไม่ได้แก้บรรทัด `st.tabs` (ก้อนที่ 2) |
| `Port 8501 is already in use` | มีเซิร์ฟเวอร์เดิมค้างอยู่ รันเซลล์ปิดเซิร์ฟเวอร์ก่อน |
| แก้โค้ดแล้วหน้าเว็บไม่เปลี่ยน | Streamlit จะถามว่า Rerun ให้กด **Always rerun** |
| ทำนายได้คำตอบเดิมตลอด | ข้อมูลตัวอย่างน้อยเกินไป ลองเพิ่มแถวใน `ข้อมูลรดน้ำ` |

---

## คำถามท้าย Lab
1. งานนี้เป็นงาน AI แบบใด (จำแนก / ทำนายตัวเลข)? เพราะอะไร
2. ถ้าอยากให้แม่นขึ้น ควรปรับปรุงข้อมูลอย่างไร?
3. การรวมทุกอย่างไว้ในเว็บเดียว (4 แท็บ) ดีกว่าแยกเป็นหลายเว็บอย่างไรสำหรับเกษตรกร?